# Single study run

## configs


In [1]:
import importlib
import copy
import gc
import json
import itertools
import torch

import engine.tuning
import utils.optuna_yaml
import utils.optuna_utils

importlib.reload(engine.tuning)
importlib.reload(utils.optuna_yaml)
importlib.reload(utils.optuna_utils)

from engine.tuning import run_optuna_study
from utils.optuna_utils import (
    build_optuna_setting_paths_from_configs,
    make_study_name,
    build_search_space_yaml_path,
    infer_channels,
    extract_top_k_trials,
    save_topk_summaries_json,
    scan_all_optuna_storages,
    scan_latest_studies_topk,
    prune_optuna_studies,
    load_json_file,
)

In [ ]:
# libraries
import os
import torch
from argparse import Namespace



configs = Namespace(
    # -- Fixed Experiment Params --
    seed=42,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    verbose=False,

    # -- Data (loop will override these) --
    data_name="ETTh1",
    window=96,
    horizon=96,
    stride=1,
    batch_size=32,
    ratios=(0.7, 0.15, 0.15),
    channels=None,

    # -- Model Identifier (loop will override these) --
    model_name="gru",                  # "gru" | "transformer" | "mamba"
    token_style="variate",             # "time" | "variate"
    

    # -- Global Searchable Hparams (YAML will override searched ones) --
    # Model common
    d_model=128,
    num_layers=2,
    dropout=0.1,

    # GRU specific
    bidirectional=True,

    # Transformer Specific
    nhead=8,
    dim_feedforward=256,
    activation="gelu",
    norm_first=False,
    layer_norm_eps=1e-5,
    batch_first=True,
    pos_emb="sin",      # only for time-token

    # Mamba Specific
    d_state=16,
    expand=2,
    d_conv=4,
    dt_rank="auto",
    dt_min=0.001,
    dt_max=0.1,
    use_fast_path=True,
    d_ff=256,           # S-D-Mamba uses a separate FFN dim

    # pooling layer (if time)
    pooling="last",
    temperature=1.0,
    

    
    # Optimization
    lr=1e-4,
    weight_decay=0.0,
    scheduler_factor=0.5,
    scheduler_patience=3,
    min_lr=1e-7,
    early_stop_patience=5,
    num_epochs=20,

    # Built automatically later
    search_space_yaml=None,    

)

# configs.channels = infer_channels(configs.data_name)
# configs.search_space_yaml = build_search_space_yaml_path(configs)
# print(configs.search_space_yaml)

## Build DB Path


In [67]:
from utils.optuna_utils import (
    build_optuna_setting_paths_from_configs,
    make_study_name,
    # list_studies_in_storage,
)


# Build setting DB path (no Optuna yet)
paths = build_optuna_setting_paths_from_configs(
    root_dir="./optuna_runs_yaml",
    configs=configs,
    create=True,
)


print(f"db_path: {paths.db_path}")
print(f"storage: {paths.storage}")

db_path: ./optuna_runs_yaml/ETTh1/gru_variate/L96_T96/study.db
storage: sqlite:////home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/optuna_runs_yaml/ETTh1/gru_variate/L96_T96/study.db


In [68]:
# Generate a simple study name
study_name = make_study_name(
    prefix="tuning_yaml",
    with_seconds=False,
)

print("study_name:", study_name)

study_name: tuning_yaml_2026-03-08_20-36


## Run Study

In [69]:
from engine.tuning import run_optuna_study


study = run_optuna_study(
    base_configs=configs,
    storage=paths.storage,   
    study_name=study_name,
    n_trials=3,
    timeout=None,
    show_progress_bar=True,
    optuna_verbosity=True,
    objective_mode="yaml",
    print_yaml=True,
)

print("Best value:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-08 20:37:10,182] A new study created in RDB with name: tuning_yaml_2026-03-08_20-36


  0%|          | 0/3 [00:00<?, ?it/s]

[YAML] configs/search_spaces/etth1_gru_variate_L96_T96.yaml | ETTh1 | gru | variate | L96_T96 | search=8
[I 2026-03-08 20:37:25,877] Trial 0 finished with value: 0.37930653348232873 and parameters: {'dropout': 0.24639655095950502, 'd_model': 256, 'num_layers': 3, 'lr': 0.00010078893141001801, 'weight_decay': 2.5375090118984246e-06, 'scheduler_factor': 0.4612187295543997, 'scheduler_patience': 3, 'bidirectional': True}. Best is trial 0 with value: 0.37930653348232873.
[I 2026-03-08 20:37:37,824] Trial 1 finished with value: 0.38050773501100865 and parameters: {'dropout': 0.09778909791528025, 'd_model': 256, 'num_layers': 3, 'lr': 0.0002245752326610148, 'weight_decay': 2.0876495603478593e-05, 'scheduler_factor': 0.4263871218902791, 'scheduler_patience': 2, 'bidirectional': True}. Best is trial 0 with value: 0.37930653348232873.
[I 2026-03-08 20:37:50,675] Trial 2 finished with value: 0.3757777825354742 and parameters: {'dropout': 0.2854594048585923, 'd_model': 128, 'num_layers': 2, 'lr':

In [ ]:
# Best value: 0.4753729169473089
# Best params: {'d_model': 256, 'num_layers': 1, 'lr': 0.0004421727214455944, 'weight_decay': 4.8902115645917925e-05, 'scheduler_factor': 0.4936725254054019, 'scheduler_patience': 4, 'bidirectional': False}

In [ ]:
# Best value: 0.4455031681710487
# Best params: {'d_model': 128, 'num_layers': 1, 'lr': 0.00025236327180901694, 'weight_decay': 5.383736593326993e-05, 'scheduler_factor': 0.5598809674993457, 'scheduler_patience': 2, 'bidirectional': True}

### Inspect DB

In [30]:
paths.storage

'sqlite:////home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/optuna_runs_yaml/ETTh1/mamba_time/L96_T96/study.db'

In [70]:
# inspect db studies
from utils.optuna_utils import list_studies_in_storage


summaries = list_studies_in_storage(
    # storage="sqlite:///optuna_runs/ETTh1/gru_time/L96_T96/study.db",
    storage=paths.storage,
    verbose=True,
)


📚 Found 1 studies in sqlite:////home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/optuna_runs_yaml/ETTh1/gru_variate/L96_T96/study.db

🔹 tuning_yaml_2026-03-08_20-36   | trials=  3 | best=0.375778        | direction=MINIMIZE


In [ ]:

# 📚 Found 4 studies in sqlite:////home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/optuna_runs/ETTh1/gru_time/L96_T96/study.db

# 🔹 tuning_2026-03-04_13-40        | trials=  3 | best=0.445503        | direction=MINIMIZE
# 🔹 tuning_2026-03-04_14-06        | trials=  3 | best=0.475373        | direction=MINIMIZE
# 🔹 tuning_2026-03-04_15-21        | trials=  3 | best=0.458625        | direction=MINIMIZE
# 🔹 tuning_2026-03-04_21-17        | trials=  7 | best=0.501347        | direction=MINIMIZE

### Load study

In [6]:
# load study
import optuna
study = optuna.load_study(
    storage="sqlite:///optuna_runs/ETTh1/gru_time/L96_T96/study.db",
    study_name="tuning_2026-03-04_13-40",
)

### explore study

In [71]:
study.best_params

{'dropout': 0.2854594048585923,
 'd_model': 128,
 'num_layers': 2,
 'lr': 0.00014428956391769515,
 'weight_decay': 4.15680800062525e-06,
 'scheduler_factor': 0.4038578586981261,
 'scheduler_patience': 4,
 'bidirectional': True}

In [10]:
# nedd to be sorted
df = study.trials_dataframe()
df.sort_values("value")

,number,value,datetime_start,datetime_complete,duration,params_bidirectional,params_d_model,params_dropout,params_lr,params_num_layers,params_scheduler_factor,params_scheduler_patience,params_weight_decay,state
1,1,0.445503,2026-03-04 13:53:36.773187,2026-03-04 13:53:53.397779,0 days 00:00:16.624592,True,128,NaN,0.000252,1,0.559881,2,0.000054,COMPLETE
0,0,0.493232,2026-03-04 13:53:02.913225,2026-03-04 13:53:36.682559,0 days 00:00:33.769334,False,256,0.126583,0.000112,2,0.431850,2,0.000035,COMPLETE
2,2,0.501223,2026-03-04 13:53:53.479414,2026-03-04 13:54:36.136434,0 days 00:00:42.657020,True,256,0.133045,0.000270,2,0.488928,3,0.000095,COMPLETE


In [3]:
# print(df.sort_values("value"))

# Run batch of studies

## Loop across data/token/model configs

In [ ]:
# --- Toggle these together ---
EXPERIMENT_MODE = "multivariate"    # Change to "increasing_lookback" later
window_horizon_pairs = [(96, 96)]      # or [(720, 96)] when ready
# -----------------------------

datasets = ["ETTh1", "ECL"]
models = ["gru", "transformer", "mamba"]
token_styles = ["time", "variate"]


all_12_configs = list(
    itertools.product(
        datasets,
        models,
        token_styles,
        window_horizon_pairs,
    )
)

print("Total configs:", len(all_12_configs))
# all_12_configs

Total configs: 12


In [4]:
# test_configs = [
#     ("ETTh1", "mamba", "time", 96, 96),
#     ("ETTh1", "transformer", "time", 96, 96),
#     ("ETTh1", "transformer", "variate", 96, 96),
#     ("ETTh1", "gru", "variate", 96, 96),
# ]

loop_topk_summaries = []


for data_name, model_name, token_style, (window, horizon) in all_12_configs:
    # 1) Fresh clean config copy
    cfg = copy.deepcopy(configs)

    # 2) Identity
    cfg.data_name = data_name
    cfg.model_name = model_name
    cfg.token_style = token_style
    cfg.window = window
    cfg.horizon = horizon

    # 3) Recompute dependent fields
    cfg.channels = infer_channels(cfg.data_name)
    cfg.search_space_yaml = build_search_space_yaml_path(cfg, mode=EXPERIMENT_MODE)

    # 4) Build storage paths
    paths = build_optuna_setting_paths_from_configs(
        root_dir="./optuna_runs_yaml",
        configs=cfg,
        create=True,
    )

    # 5) New study name each time
    study_name = make_study_name(
        prefix="tuning_yaml",
        with_seconds=False,
    )
    
    print("\n" + "=" * 90)
    print(
        f"RUNNING: {cfg.data_name} | {cfg.model_name} | {cfg.token_style} "
        f"| L{cfg.window}_T{cfg.horizon}"
    )
    print(f"db_path: {paths.db_path}")
    print(f"study_name: {study_name}")

    # 6) Run study
    study = run_optuna_study(
        base_configs=cfg,
        storage=paths.storage,
        study_name=study_name,
        n_trials=30,
        timeout=None,
        show_progress_bar=True,
        optuna_verbosity=False,
        objective_mode="yaml",
        print_yaml=True,
    )

    print("Best value:", study.best_value)
    print("Best params:", study.best_params)

    # 7) Quietly collect compact top-k summary
    summary = extract_top_k_trials(
        study=study,
        yaml_path=cfg.search_space_yaml,
        db_path=paths.db_path,
        k=1,
    )
    loop_topk_summaries.append(summary)


    # 8) Cleanup before next config
    del summary
    del study
    del cfg
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Save json summaries once the loop finishes
save_topk_summaries_json(
    summaries=loop_topk_summaries,
    save=True,
)



RUNNING: ETTh1 | gru | time | L96_T96
db_path: ./optuna_runs_yaml/ETTh1/gru_time/L96_T96/study.db
study_name: tuning_yaml_2026-03-10_18-58


  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/etth1_gru_time_L96_T96.yaml | ETTh1 | gru | time | L96_T96 | search=10
Best value: 0.45338814333388866
Best params: {'lr': 0.00018128731151278876, 'weight_decay': 1.3857247424345708e-06, 'd_model': 256, 'num_layers': 1, 'pooling': 'last', 'bidirectional': True, 'dropout': 0.12000870613177515, 'scheduler_factor': 0.4721505131676175, 'scheduler_patience': 3}

RUNNING: ETTh1 | gru | variate | L96_T96
db_path: ./optuna_runs_yaml/ETTh1/gru_variate/L96_T96/study.db
study_name: tuning_yaml_2026-03-10_19-09


  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/etth1_gru_variate_L96_T96.yaml | ETTh1 | gru | variate | L96_T96 | search=8
Best value: 0.3706525045421673
Best params: {'lr': 0.0005414759960562863, 'weight_decay': 6.411343629247723e-06, 'd_model': 256, 'num_layers': 1, 'bidirectional': True, 'dropout': 0.0973160013870552, 'scheduler_factor': 0.5340626250455648, 'scheduler_patience': 5}

RUNNING: ETTh1 | transformer | time | L96_T96
db_path: ./optuna_runs_yaml/ETTh1/transformer_time/L96_T96/study.db
study_name: tuning_yaml_2026-03-10_19-14


...

  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/ecl_transformer_variate_L96_T96.yaml | ECL | transformer | variate | L96_T96 | search=11
Best value: 0.12583328862085483
Best params: {'lr': 0.0006189127560854512, 'weight_decay': 1.39037839376548e-06, 'd_model': 512, 'num_layers': 3, 'nhead': 8, 'dim_feedforward': 1024, 'activation': 'relu', 'norm_first': True, 'dropout': 0.07412356604180327, 'scheduler_factor': 0.5318339030906468, 'scheduler_patience': 5}

RUNNING: ECL | mamba | time | L96_T96
db_path: ./optuna_runs_yaml/ECL/mamba_time/L96_T96/study.db
study_name: tuning_yaml_2026-03-11_08-32


  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/ecl_mamba_time_L96_T96.yaml | ECL | mamba | time | L96_T96 | search=14
[W 2026-03-11 08:36:08,406] Trial 3 failed with parameters: {'lr': 0.00030096352894207207, 'weight_decay': 4.381911119298253e-06, 'd_model': 256, 'num_layers': 3, 'd_state': 8, 'expand': 1, 'd_conv': 8, 'd_ff': 128, 'pooling': 'attention', 'bidirectional': True, 'dropout': 0.2117506707750587, 'scheduler_factor': 0.5466654458968626, 'scheduler_patience': 4, 'temperature': 1.223658093749623} because of the following error: RuntimeError('causal_conv1d only supports width between 2 and 4').
Traceback (most recent call last):
  File "/home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/.tsf_venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/engine/tuning.py", line 359, in <lambda>
    obj_fn = lambda trial: objecti

RuntimeError: causal_conv1d only supports width between 2 and 4

## Minimal recovery loop (run only the missing ones)

In [10]:
import importlib
import engine.tuning
importlib.reload(engine.tuning)
from engine.tuning import run_optuna_study

In [11]:
# --- recovery configs ---
EXPERIMENT_MODE = "multivariate"
recovery_configs = [
    ("ECL", "mamba", "time", (96, 96)),
    ("ECL", "mamba", "variate", (96, 96)),
]

loop_topk_summaries2 = []



for data_name, model_name, token_style, (window, horizon) in recovery_configs:

    cfg = copy.deepcopy(configs)

    cfg.data_name = data_name
    cfg.model_name = model_name
    cfg.token_style = token_style
    cfg.window = window
    cfg.horizon = horizon

    cfg.channels = infer_channels(cfg.data_name)
    cfg.search_space_yaml = build_search_space_yaml_path(cfg, mode=EXPERIMENT_MODE)

    paths = build_optuna_setting_paths_from_configs(
        root_dir="./optuna_runs_yaml",
        configs=cfg,
        create=True,
    )

    study_name = make_study_name(
        prefix="tuning_yaml",
        with_seconds=False,
    )

    print("\n" + "=" * 90)
    print(
        f"RECOVERY RUN: {cfg.data_name} | {cfg.model_name} | {cfg.token_style} "
        f"| L{cfg.window}_T{cfg.horizon}"
    )

    study = run_optuna_study(
        base_configs=cfg,
        storage=paths.storage,
        study_name=study_name,
        n_trials=30,
        timeout=None,
        show_progress_bar=True,
        optuna_verbosity=False,
        objective_mode="yaml",
        print_yaml=True,
    )

    print("Best value:", study.best_value)
    print("Best params:", study.best_params)

    summary = extract_top_k_trials(
        study=study,
        yaml_path=cfg.search_space_yaml,
        db_path=paths.db_path,
        k=1,
    )

    loop_topk_summaries2.append(summary)
    
    del summary
    del study
    del cfg
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


save_topk_summaries_json(
    summaries=loop_topk_summaries2,
    save=True,
)


RECOVERY RUN: ECL | mamba | time | L96_T96


  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/ecl_mamba_time_L96_T96.yaml | ECL | mamba | time | L96_T96 | search=14
Best value: 0.21213497119483238
Best params: {'lr': 0.00048028785755296584, 'weight_decay': 3.879783712847491e-06, 'd_model': 512, 'num_layers': 3, 'd_state': 8, 'expand': 2, 'd_conv': 4, 'd_ff': 128, 'pooling': 'attention', 'bidirectional': True, 'dropout': 0.2599811216001546, 'scheduler_factor': 0.5096496496760864, 'scheduler_patience': 4, 'temperature': 1.556272764196267}

RECOVERY RUN: ECL | mamba | variate | L96_T96


  0%|          | 0/30 [00:00<?, ?it/s]

[YAML] configs/search_spaces/multivariate/ecl_mamba_variate_L96_T96.yaml | ECL | mamba | variate | L96_T96 | search=11
Best value: 0.12710743603233332
Best params: {'lr': 0.0006655285592493306, 'weight_decay': 2.8146570532571993e-06, 'd_model': 256, 'num_layers': 1, 'd_state': 8, 'expand': 2, 'd_conv': 2, 'd_ff': 512, 'dropout': 0.20883419105923154, 'scheduler_factor': 0.4318950547834703, 'scheduler_patience': 2}

💾 Top-k summaries saved to:
./optuna_runs_yaml/summaries/topk_summary_2026-03-11_14-18.json


'./optuna_runs_yaml/summaries/topk_summary_2026-03-11_14-18.json'

### clean databases

In [5]:
prune_optuna_studies(
    root_dir="./optuna_runs_yaml",
    keep_top_n=2,
    prune=True,
    verbose=False,
)

15

In [1]:
cleaned_summary, _ = scan_all_optuna_storages(
    root_dir="./optuna_runs_yaml",
    save=False,
    verbose=True,
)

NameError: name 'scan_all_optuna_storages' is not defined

### inspect the collected summaries

#### latest top_k summaries

In [4]:
latest_topk_summaries, latest_topk_json_path = scan_latest_studies_topk(
    root_dir="./optuna_runs_yaml",
    yaml_root_dir="configs/search_spaces",
    filename_prefix="latest_topk_summary",
    k=1,
    save=False,
    verbose=True,
)

🔎 Found 12 study.db files under ./optuna_runs_yaml
📚 Found 2 studies in sqlite:///./optuna_runs_yaml/ECL/gru_time/L96_T96/study.db
🔹 tuning_yaml_2026-03-10_03-57   | trials= 20 | best=0.199398        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-10_20-05   | trials= 30 | best=0.193495        | direction=MINIMIZE
✅ Latest study for ECL/gru_time/L96_T96 -> tuning_yaml_2026-03-10_20-05 | best=0.193495
📚 Found 2 studies in sqlite:///./optuna_runs_yaml/ECL/gru_variate/L96_T96/study.db
🔹 tuning_yaml_2026-03-10_04-33   | trials= 20 | best=0.126651        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-10_21-12   | trials= 30 | best=0.126233        | direction=MINIMIZE
✅ Latest study for ECL/gru_variate/L96_T96 -> tuning_yaml_2026-03-10_21-12 | best=0.126233
📚 Found 3 studies in sqlite:///./optuna_runs_yaml/ECL/mamba_time/L96_T96/study.db
🔹 tuning_yaml_2026-03-10_09-42   | trials= 20 | best=0.214657        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-11_08-32   | trials=  4 | best=0.230113        | dire

#### post scan summary

In [3]:
post_scan_summary, post_scan_json_path = scan_all_optuna_storages(
    root_dir="./optuna_runs_yaml",
    filename_prefix="scan_summary",
    save=True,   # change to True when ready
    verbose=False,
)

In [38]:
# post_scan_summary

#### load json

In [43]:
topk_path = "optuna_runs_yaml/summaries/latest_topk_summary_2026-03-10_00-13.json"
topk_summary = load_json_file(json_path=topk_path, verbose=False,)

In [52]:
len(topk_summary)


12

### inspect DB

In [7]:
# inspect db studies
from utils.optuna_utils import list_studies_in_storage

""
summaries = list_studies_in_storage(
    storage="sqlite:///optuna_runs_yaml/ETTh1/mamba_time/L96_T96/study.db",
    # storage=paths.storage,
    verbose=True,
)

📚 Found 2 studies in sqlite:///optuna_runs_yaml/ETTh1/mamba_time/L96_T96/study.db
🔹 tuning_yaml_2026-03-08_18-38   | trials=  3 | best=0.486698        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-08_22-44   | trials=  3 | best=0.515395        | direction=MINIMIZE


In [ ]:
scan_summary, scan_json_path = scan_all_optuna_storages(
    root_dir="./optuna_runs_yaml",
    save=False,
    verbose=True,
)

🔎 Found 4 study.db files under ./optuna_runs_yaml
📚 Found 3 studies in sqlite:///./optuna_runs_yaml/ETTh1/gru_variate/L96_T96/study.db
🔹 tuning_yaml_2026-03-08_20-36   | trials=  3 | best=0.375778        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-08_22-48   | trials=  3 | best=0.370173        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-09_00-41   | trials=  3 | best=0.372977        | direction=MINIMIZE
📚 Found 5 studies in sqlite:///./optuna_runs_yaml/ETTh1/mamba_time/L96_T96/study.db
🔹 tuning_yaml_2026-03-08_17-15   | trials=  1 | best=0.698539        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-08_17-58   | trials=  2 | best=0.583737        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-08_18-38   | trials=  3 | best=0.486698        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-08_22-44   | trials=  3 | best=0.515395        | direction=MINIMIZE
🔹 tuning_yaml_2026-03-09_00-38   | trials=  3 | best=0.558574        | direction=MINIMIZE
📚 Found 3 studies in sqlite:///./optuna_runs_yaml/ETTh1/trans